# ANP
---
_**A**nalytic **N**etwork **P**rocess_

In [1]:
import numpy as np
import pandas as pd

## 1. Introduction
---
The **ANP** (_Analytic Network Process_) method derives priorities from pairwise comparisons in a **network**. It extends [AHP](./ahp.ipynb) by allowing **dependence within clusters** and **feedback between clusters**. Local priorities are combined in a **supermatrix**, whose limiting values give the network priorities.

### 1.1. Applications
---
The _ANP method_ supports decisions where the available choices and their evaluation criteria influence each other, such as:

- Infrastructure investment
- Healthcare service improvement
- Technology selection
- Environmental planning

Consider a municipality choosing a **flood mitigation project**. **Protection**, **affordability over the project lifetime**, and **ecology** interact: ecological condition affects water retention and maintenance, while the proposed projects change which considerations matter most. All project descriptions and judgments below are **illustrative**, rather than observations from a particular municipality.

In [2]:
df = pd.DataFrame({
    "project": [
        "Create a retention basin and restore wetland habitat",
        "Enlarge an existing stormwater culvert",
        "Build a flood wall along the river"
    ],
    "main tradeoff": [
        "Land and maintenance requirements",
        "Limited ecological benefits",
        "High lifecycle cost and displaced flood risk"
    ]
}, index=["Retention basin", "Larger culvert", "Flood wall"])

df

,project,main tradeoff
Retention basin,Create a retention basin and restore wetland h...,Land and maintenance requirements
Larger culvert,Enlarge an existing stormwater culvert,Limited ecological benefits
Flood wall,Build a flood wall along the river,High lifecycle cost and displaced flood risk


## 2. Method steps
---

### 2.1. Network structure
---
Use two clusters: **Criteria** and **Alternatives**, under the control criterion **overall suitability for flood mitigation**. Here, a supermatrix entry in **row** $i$, **column** $j$ measures the relative influence of element $i$ **with respect to** element $j$.

| block | rows | columns | comparison question |
| :---: | :--- | :--- | :--- |
| $D$ | Criteria | Criteria | Which criterion has more influence on this criterion? |
| $P$ | Alternatives | Criteria | Which project contributes more to this criterion? |
| $F$ | Criteria | Alternatives | Which criterion matters more when evaluating this project? |

Thus $D$ represents **inner dependence**, and $P$ together with $F$ creates **feedback**. No direct dependence between projects is assumed; its block is zero. A criterion's direct influence on itself is retained in $D$.

In [3]:
CRITERIA = ["Protection", "Affordability", "Ecology"]
ALTERNATIVES = list(df.index)
CLUSTERS = {
    "Criteria": CRITERIA,
    "Alternatives": ALTERNATIVES
}

NODES = CRITERIA + ALTERNATIVES

### 2.2. Pairwise comparison matrices
---
For each column element and each influencing cluster, compare the row elements using a **positive reciprocal matrix** $M=[a_{ij}]$:

$$ \large
a_{ii}=1, \qquad a_{ji}=\frac{1}{a_{ij}}
$$

The usual scale assigns **1** to equal influence, **3** to moderate influence, **5** to strong influence, **7** to very strong influence, and **9** to extreme influence; **2, 4, 6, 8** are intermediate judgments.

First compare projects in the order shown in `ALTERNATIVES`. Under **Protection**, for example, the retention basin is judged **twice** as preferable as the larger culvert and **four times** as preferable as the flood wall. Affordability favors the culvert; ecology strongly favors the basin. These are elicited preference ratios, rather than ratios of construction costs or flood volumes.

In [4]:
PWCM_P = {
    "Protection": pd.DataFrame([
        [1  , 2  , 4],
        [1/2, 1  , 2],
        [1/4, 1/2, 1]
    ], index=ALTERNATIVES, columns=ALTERNATIVES),
    "Affordability": pd.DataFrame([
        [1  , 1/2, 2],
        [2  , 1  , 4],
        [1/2, 1/4, 1]
    ], index=ALTERNATIVES, columns=ALTERNATIVES),
    "Ecology": pd.DataFrame([
        [1  , 8  , 4],
        [1/8, 1  , 1/2],
        [1/4, 2  , 1]
    ], index=ALTERNATIVES, columns=ALTERNATIVES)
}

pd.concat(PWCM_P)

Retention basin  Larger culvert  Flood wall
Protection    Retention basin            1.000            2.00         4.0
              Larger culvert             0.500            1.00         2.0
              Flood wall                 0.250            0.50         1.0
Affordability Retention basin            1.000            0.50         2.0
              Larger culvert             2.000            1.00         4.0
              Flood wall                 0.500            0.25         1.0
Ecology       Retention basin            1.000            8.00         4.0
              Larger culvert             0.125            1.00         0.5
              Flood wall                 0.250            2.00         1.0

Next compare criteria in the order shown in `CRITERIA`. In $D$, ecological condition is judged to have the greatest influence on water retention, lifecycle affordability, and ecological performance itself. In $F$, protection and ecology are equally important for the basin, while protection is dominant for the culvert and wall. Each matrix is conditioned on the column element named in its key.

In [5]:
PWCM_D = {
    "Protection": pd.DataFrame([
        [1  , 2  , 1/2],
        [1/2, 1  , 1/4],
        [2  , 4  , 1]
    ], index=CRITERIA, columns=CRITERIA),
    "Affordability": pd.DataFrame([
        [1  , 1/2, 1/4],
        [2  , 1  , 1/2],
        [4  , 2  , 1]
    ], index=CRITERIA, columns=CRITERIA),
    "Ecology": pd.DataFrame([
        [1  , 1  , 1/4],
        [1  , 1  , 1/4],
        [4  , 4  , 1]
    ], index=CRITERIA, columns=CRITERIA)
}

PWCM_F = {
    "Retention basin": pd.DataFrame([
        [1  , 2  , 1],
        [1/2, 1  , 1/2],
        [1  , 2  , 1]
    ], index=CRITERIA, columns=CRITERIA),
    "Larger culvert": pd.DataFrame([
        [1  , 2  , 4],
        [1/2, 1  , 2],
        [1/4, 1/2, 1]
    ], index=CRITERIA, columns=CRITERIA),
    "Flood wall": pd.DataFrame([
        [1  , 4  , 4],
        [1/4, 1  , 1],
        [1/4, 1  , 1]
    ], index=CRITERIA, columns=CRITERIA)
}

pd.concat({"D": pd.concat(PWCM_D), "F": pd.concat(PWCM_F)})

Protection  Affordability  Ecology
D Protection      Protection           1.00            2.0     0.50
                  Affordability        0.50            1.0     0.25
                  Ecology              2.00            4.0     1.00
  Affordability   Protection           1.00            0.5     0.25
                  Affordability        2.00            1.0     0.50
                  Ecology              4.00            2.0     1.00
  Ecology         Protection           1.00            1.0     0.25
                  Affordability        1.00            1.0     0.25
                  Ecology              4.00            4.0     1.00
F Retention basin Protection           1.00            2.0     1.00
                  Affordability        0.50            1.0     0.50
                  Ecology              1.00            2.0     1.00
  Larger culvert  Protection           1.00            2.0     4.00
                  Affordability        0.50            1.0     2.00
                  Ecology              0.25            0.5     1.00
  Flood wall      Protection           1.00            4.0     4.00
                  Affordability        0.25            1.0     1.00
                  Ecology              0.25            1.0     1.00

### 2.3. Local priorities and consistency
---
For **every** comparison matrix, calculate its principal right eigenvector and normalize it:

$$ \large
Mv=\lambda_{\max}v, \qquad w_i=\frac{v_i}{\sum_j v_j}
$$

Check the **consistency ratio** before using the judgments:

$$ \large
CI=\frac{\lambda_{\max}-n}{n-1}, \qquad CR=\frac{CI}{RI_n}
$$

Here all element comparisons have order **3**, with $RI_3=0.58$. Reciprocal matrices of order **2** are always consistent, so their $CR$ is set to zero. A $CR$ below **0.10** is commonly accepted; this checks coherence, rather than the empirical validity of the judgments. The main example uses internally consistent judgments to make the calculations easy to follow.

In [6]:
def local_priorities(matrix):
    values = matrix.to_numpy(dtype=float)
    if not (np.all(values > 0) and np.allclose(values * values.T, 1)):
        raise ValueError("Pairwise judgments must be positive and reciprocal.")

    eigenvalues, eigenvectors = np.linalg.eig(values)
    principal_idx = np.argmax(eigenvalues.real)
    lambda_max = float(eigenvalues[principal_idx].real)
    vector = eigenvectors[:, principal_idx].real
    weights = pd.Series(vector/vector.sum(), index=matrix.index)

    n = len(matrix)
    CI = max(0, (lambda_max - n)/(n - 1))
    CR = CI/0.58 if n == 3 else 0
    if n not in (2, 3):
        raise ValueError("This example supports comparison orders 2 and 3.")
    if CR >= 0.1:
        raise ValueError("Consistency ratio not acceptable; review judgments.")

    return weights, lambda_max, CI, CR


LOCAL_PRIORITIES = {}
consistency = []

for block, matrices in {"D": PWCM_D, "P": PWCM_P, "F": PWCM_F}.items():
    for node, matrix in matrices.items():
        weights, lambda_max, CI, CR = local_priorities(matrix)
        LOCAL_PRIORITIES[(block, node)] = weights
        consistency.append([block, node, lambda_max, CI, CR])

df_consistency = pd.DataFrame(
    consistency, columns=["block", "column element", "lambda_max", "CI", "CR"]
)

df_consistency

,block,column element,lambda_max,CI,CR
0,D,Protection,3.0,0,0.0
1,D,Affordability,3.0,0,0.0
2,D,Ecology,3.0,0,0.0
3,P,Protection,3.0,0,0.0
4,P,Affordability,3.0,0,0.0
5,P,Ecology,3.0,0,0.0
6,F,Retention basin,3.0,0,0.0
7,F,Larger culvert,3.0,0,0.0
8,F,Flood wall,3.0,0,0.0


In [7]:
D = pd.DataFrame({
    node: LOCAL_PRIORITIES[("D", node)] for node in CRITERIA
})
P = pd.DataFrame({
    node: LOCAL_PRIORITIES[("P", node)] for node in CRITERIA
})
F = pd.DataFrame({
    node: LOCAL_PRIORITIES[("F", node)] for node in ALTERNATIVES
})

# Every local priority column sums to one within its influencing cluster.
pd.concat({"D": D, "P": P, "F": F}, axis=0).fillna(0)

Protection  Affordability   Ecology  Retention basin  \
D Protection         0.285714       0.142857  0.166667              0.0   
  Affordability      0.142857       0.285714  0.166667              0.0   
  Ecology            0.571429       0.571429  0.666667              0.0   
P Retention basin    0.571429       0.285714  0.727273              0.0   
  Larger culvert     0.285714       0.571429  0.090909              0.0   
  Flood wall         0.142857       0.142857  0.181818              0.0   
F Protection         0.000000       0.000000  0.000000              0.4   
  Affordability      0.000000       0.000000  0.000000              0.2   
  Ecology            0.000000       0.000000  0.000000              0.4   

                   Larger culvert  Flood wall  
D Protection             0.000000    0.000000  
  Affordability          0.000000    0.000000  
  Ecology                0.000000    0.000000  
P Retention basin        0.000000    0.000000  
  Larger culvert         0.000000    0.000000  
  Flood wall             0.000000    0.000000  
F Protection             0.571429    0.666667  
  Affordability          0.285714    0.166667  
  Ecology                0.142857    0.166667

### 2.4. Unweighted supermatrix
---
Place the local priority vectors in their corresponding blocks:

$$ \large
W=\begin{bmatrix}D&F\\P&0\end{bmatrix}
$$

Criteria columns contain **two** normalized blocks, so their total is **2**. Alternative columns contain one block, so their total is **1**. Consequently, this matrix still needs **cluster weights**.

In [8]:
W = pd.DataFrame(0.0, index=NODES, columns=NODES)
W.loc[CRITERIA, CRITERIA] = D
W.loc[ALTERNATIVES, CRITERIA] = P
W.loc[CRITERIA, ALTERNATIVES] = F

pd.concat([W, W.sum().to_frame("column sum").T])

,Protection,Affordability,Ecology,Retention basin,Larger culvert,Flood wall
Protection,0.285714,0.142857,0.166667,0.4,0.571429,0.666667
Affordability,0.142857,0.285714,0.166667,0.2,0.285714,0.166667
Ecology,0.571429,0.571429,0.666667,0.4,0.142857,0.166667
Retention basin,0.571429,0.285714,0.727273,0.0,0.000000,0.000000
Larger culvert,0.285714,0.571429,0.090909,0.0,0.000000,0.000000
Flood wall,0.142857,0.142857,0.181818,0.0,0.000000,0.000000
column sum,2.000000,2.000000,2.000000,1.0,1.000000,1.000000


### 2.5. Cluster weights and weighted supermatrix
---
For a **Criteria** column, compare the influence of the Criteria and Alternatives clusters: the panel judges the former **twice** as influential, giving weights $2/3$ and $1/3$. The same cluster judgment is assumed for all three criteria. For an **Alternatives** column, only Criteria has influence, so its weight is **1**.

Multiply each block $W_{ab}$ by its cluster weight $g_{ab}$:

$$ \large
\overline{W}_{ab}=g_{ab}W_{ab}, \qquad \sum_i\overline{w}_{ij}=1
$$

The cluster weights are additional judgments, rather than arbitrary normalization of the full columns.

In [9]:
PWCM_CLUSTER = pd.DataFrame([
    [1  , 2],
    [1/2, 1]
], index=CLUSTERS, columns=CLUSTERS)

cluster_weights, _, _, cluster_CR = local_priorities(PWCM_CLUSTER)

G = pd.DataFrame(0.0, index=CLUSTERS, columns=CLUSTERS)
G["Criteria"] = cluster_weights
G.loc["Criteria", "Alternatives"] = 1

G

,Criteria,Alternatives
Criteria,0.666667,1.0
Alternatives,0.333333,0.0


In [10]:
W_w = W.copy()

for row_cluster, rows in CLUSTERS.items():
    for col_cluster, cols in CLUSTERS.items():
        W_w.loc[rows, cols] *= G.loc[row_cluster, col_cluster]

assert np.allclose(W_w.sum(), 1)

pd.concat([W_w, W_w.sum().to_frame("column sum").T])

,Protection,Affordability,Ecology,Retention basin,Larger culvert,Flood wall
Protection,0.190476,0.095238,0.111111,0.4,0.571429,0.666667
Affordability,0.095238,0.190476,0.111111,0.2,0.285714,0.166667
Ecology,0.380952,0.380952,0.444444,0.4,0.142857,0.166667
Retention basin,0.190476,0.095238,0.242424,0.0,0.000000,0.000000
Larger culvert,0.095238,0.190476,0.030303,0.0,0.000000,0.000000
Flood wall,0.047619,0.047619,0.060606,0.0,0.000000,0.000000
column sum,1.000000,1.000000,1.000000,1.0,1.000000,1.000000


### 2.6. Limit supermatrix
---
Raise the weighted supermatrix to successive powers:

$$ \large
L=\lim_{k\to\infty}\overline{W}^{\,k}
$$

In this network, every node is reachable from every other node, and the criteria have self-influence. In fact, $\overline{W}^{\,2}$ is strictly positive: the matrix is **primitive**, so its powers converge to identical columns. Column sums equal to one alone do **not** guarantee this convergence for arbitrary ANP networks; periodic or reducible networks need separate treatment.

Stop when the largest absolute change is below $10^{-12}$, with a maximum of **10,000** iterations.

In [11]:
values = W_w.to_numpy()
assert np.all(values @ values > 0)

L = values.copy()

for k in range(2, 10001):
    L_next = values @ L
    error = np.max(np.abs(L_next - L))
    L = L_next
    if error < 1e-12:
        break
else:
    raise RuntimeError("The supermatrix powers did not converge.")

assert np.allclose(L, L[:, [0]], atol=1e-10, rtol=0)
assert np.allclose(values @ L, L, atol=1e-10, rtol=0)

L = pd.DataFrame(L, index=NODES, columns=NODES)

print(f'Power: {k}; largest change: {error:.2e}')
L.style.background_gradient()

Power: 27; largest change: 3.86e-13


,Protection,Affordability,Ecology,Retention basin,Larger culvert,Flood wall
Protection,0.219646,0.219646,0.219646,0.219646,0.219646,0.219646
Affordability,0.145175,0.145175,0.145175,0.145175,0.145175,0.145175
Ecology,0.385178,0.385178,0.385178,0.385178,0.385178,0.385178
Retention basin,0.149040,0.149040,0.149040,0.149040,0.149040,0.149040
Larger culvert,0.060243,0.060243,0.060243,0.060243,0.060243,0.060243
Flood wall,0.040717,0.040717,0.040717,0.040717,0.040717,0.040717


### 2.7. Priorities and ranking
---
Any column of $L$ gives the limiting priorities of **all nodes**. To compare projects, normalize the alternative entries **within their own cluster**:

$$ \large
s_i=\frac{L_{ij}}{\sum_{r\in\mathrm{Alternatives}}L_{rj}}
$$

Rank by **decreasing priority**. These priorities express preference under the modeled judgments; they are not probabilities of project success.

In [12]:
limit_priorities = L.iloc[:, 0]

CRITERIA_WEIGHTS = limit_priorities.loc[CRITERIA]
CRITERIA_WEIGHTS = CRITERIA_WEIGHTS/CRITERIA_WEIGHTS.sum()

df_rank = limit_priorities.loc[ALTERNATIVES].to_frame("limit priority")
df_rank["score"] = df_rank["limit priority"]/df_rank["limit priority"].sum()
df_rank["ranking"] = df_rank["score"].rank(ascending=False, method="dense").astype(int)

df_rank.sort_values("ranking").style.background_gradient(subset=["score"])

,limit priority,score,ranking
Retention basin,0.149040,0.596161,1
Larger culvert,0.060243,0.240973,2
Flood wall,0.040717,0.162866,3


The **retention basin** is the clear first choice, with a priority of approximately **0.596**, followed by the **larger culvert** (**0.241**) and **flood wall** (**0.163**). Its strong protection and ecological contribution outweigh the culvert's affordability advantage.

### 2.8. Effect of inner dependence
---
For comparison, replace $D$ by the identity matrix: criteria retain their own influence but no longer influence one another. Keep **all other judgments and cluster weights unchanged**, including feedback through $F$. This isolates the effect of the modeled dependence **between criteria**, rather than switching to a different scoring method.

In [13]:
W_without = W_w.copy()
W_without.loc[CRITERIA, CRITERIA] = np.eye(len(CRITERIA))*G.loc["Criteria", "Criteria"]

values_without = W_without.to_numpy()
L_without = values_without.copy()

for k_without in range(2, 10001):
    L_next = values_without @ L_without
    error_without = np.max(np.abs(L_next - L_without))
    L_without = L_next
    if error_without < 1e-12:
        break
else:
    raise RuntimeError("The comparison network did not converge.")

score_without = pd.Series(L_without[len(CRITERIA):, 0], index=ALTERNATIVES)
score_without = score_without/score_without.sum()

df_comparison = pd.DataFrame({
    "Without inner dependence": score_without,
    "With inner dependence": df_rank["score"]
})
df_comparison["change"] = df_comparison["With inner dependence"] - df_comparison["Without inner dependence"]

df_comparison.style.background_gradient(subset=list(df_comparison.columns[:2]))

,Without inner dependence,With inner dependence,change
Retention basin,0.553617,0.596161,0.042544
Larger culvert,0.292271,0.240973,-0.051298
Flood wall,0.154112,0.162866,0.008754


The basin remains first, but its priority increases from about **0.554** to **0.596**. Accounting for ecological influence strengthens its advantage. The difference follows from the explicit dependence judgments in $D$; it is not an automatic benefit assigned to environmental projects.

## 3. Examples
---

### 3.1. Hospital service improvement
---
A hospital compares **digital triage**, **triage training**, and an **extra shift** using **Safety**, **Capacity**, and **Affordability**. Safety influences usable capacity; each intervention also changes the relative relevance of these criteria. The judgments are **illustrative** and assume that the digital system has already passed clinical and regulatory assessment.

Use the same network structure. The matrices below follow the declared order of criteria or alternatives. Digital triage leads on safety and capacity; training is most affordable. The Criteria cluster has weight **1/4** and Alternatives **3/4** in criteria columns, obtained by judging Alternatives three times as influential. Alternative columns again depend only on Criteria.

In [14]:
CRITERIA_ex1 = ["Safety", "Capacity", "Affordability"]
ALTERNATIVES_ex1 = ["Digital triage", "Triage training", "Extra shift"]
NODES_ex1 = CRITERIA_ex1 + ALTERNATIVES_ex1

PWCM_P_ex1 = {
    "Safety": [[1, 3, 4], [1/3, 1, 2], [1/4, 1/2, 1]],
    "Capacity": [[1, 4, 2], [1/4, 1, 1/2], [1/2, 2, 1]],
    "Affordability": [[1, 1/4, 1/2], [4, 1, 2], [2, 1/2, 1]]
}
PWCM_D_ex1 = {
    "Safety": [[1, 2, 4], [1/2, 1, 2], [1/4, 1/2, 1]],
    "Capacity": [[1, 2, 4], [1/2, 1, 2], [1/4, 1/2, 1]],
    "Affordability": [[1, 2, 1/2], [1/2, 1, 1/4], [2, 4, 1]]
}
PWCM_F_ex1 = {
    "Digital triage": [[1, 2, 4], [1/2, 1, 2], [1/4, 1/2, 1]],
    "Triage training": [[1, 4, 2], [1/4, 1, 1/2], [1/2, 2, 1]],
    "Extra shift": [[1, 1/2, 2], [2, 1, 4], [1/2, 1/4, 1]]
}

# Local priorities and consistency for every matrix
PRIORITIES_ex1 = {}
consistency_ex1 = []

for block, matrices in {"D": PWCM_D_ex1, "P": PWCM_P_ex1, "F": PWCM_F_ex1}.items():
    labels = ALTERNATIVES_ex1 if block == "P" else CRITERIA_ex1
    for node, data in matrices.items():
        matrix = pd.DataFrame(data, index=labels, columns=labels)
        weights, lambda_max, CI, CR = local_priorities(matrix)
        PRIORITIES_ex1[(block, node)] = weights
        consistency_ex1.append([block, node, lambda_max, CI, CR])

pd.DataFrame(consistency_ex1, columns=["block", "column element", "lambda_max", "CI", "CR"])

,block,column element,lambda_max,CI,CR
0,D,Safety,3.000000,0.000000,0.000000
1,D,Capacity,3.000000,0.000000,0.000000
2,D,Affordability,3.000000,0.000000,0.000000
3,P,Safety,3.018295,0.009147,0.015771
4,P,Capacity,3.000000,0.000000,0.000000
5,P,Affordability,3.000000,0.000000,0.000000
6,F,Digital triage,3.000000,0.000000,0.000000
7,F,Triage training,3.000000,0.000000,0.000000
8,F,Extra shift,3.000000,0.000000,0.000000


In [15]:
# Unweighted supermatrix
D_ex1 = pd.DataFrame({node: PRIORITIES_ex1[("D", node)] for node in CRITERIA_ex1})
P_ex1 = pd.DataFrame({node: PRIORITIES_ex1[("P", node)] for node in CRITERIA_ex1})
F_ex1 = pd.DataFrame({node: PRIORITIES_ex1[("F", node)] for node in ALTERNATIVES_ex1})

W_ex1 = pd.DataFrame(0.0, index=NODES_ex1, columns=NODES_ex1)
W_ex1.loc[CRITERIA_ex1, CRITERIA_ex1] = D_ex1
W_ex1.loc[ALTERNATIVES_ex1, CRITERIA_ex1] = P_ex1
W_ex1.loc[CRITERIA_ex1, ALTERNATIVES_ex1] = F_ex1

# Cluster judgments and weighted supermatrix
PWCM_CLUSTER_ex1 = pd.DataFrame([
    [1, 1/3], [3, 1]
], index=CLUSTERS, columns=CLUSTERS)
cluster_weights_ex1, _, _, cluster_CR_ex1 = local_priorities(PWCM_CLUSTER_ex1)

W_ex1_w = W_ex1.copy()
W_ex1_w.loc[CRITERIA_ex1, CRITERIA_ex1] *= cluster_weights_ex1["Criteria"]
W_ex1_w.loc[ALTERNATIVES_ex1, CRITERIA_ex1] *= cluster_weights_ex1["Alternatives"]

assert np.allclose(W_ex1_w.sum(), 1)

W_ex1_w.style.background_gradient()

,Safety,Capacity,Affordability,Digital triage,Triage training,Extra shift
Safety,0.142857,0.142857,0.071429,0.571429,0.571429,0.285714
Capacity,0.071429,0.071429,0.035714,0.285714,0.142857,0.571429
Affordability,0.035714,0.035714,0.142857,0.142857,0.285714,0.142857
Digital triage,0.468760,0.428571,0.107143,0.000000,0.000000,0.000000
Triage training,0.178865,0.107143,0.428571,0.000000,0.000000,0.000000
Extra shift,0.102375,0.214286,0.214286,0.000000,0.000000,0.000000


In [16]:
# Limit supermatrix
values_ex1 = W_ex1_w.to_numpy()
assert np.all(values_ex1 @ values_ex1 > 0)

L_ex1 = values_ex1.copy()

for k_ex1 in range(2, 10001):
    L_next = values_ex1 @ L_ex1
    error_ex1 = np.max(np.abs(L_next - L_ex1))
    L_ex1 = L_next
    if error_ex1 < 1e-12:
        break
else:
    raise RuntimeError("The hospital network did not converge.")

assert np.allclose(L_ex1, L_ex1[:, [0]], atol=1e-10, rtol=0)

# Within-cluster priorities and ranking
df_ex1 = pd.DataFrame({
    "limit priority": L_ex1[len(CRITERIA_ex1):, 0]
}, index=ALTERNATIVES_ex1)
df_ex1["score"] = df_ex1["limit priority"]/df_ex1["limit priority"].sum()
df_ex1["ranking"] = df_ex1["score"].rank(ascending=False, method="dense").astype(int)

print(f'Power: {k_ex1}; largest change: {error_ex1:.2e}')
df_ex1.sort_values("ranking").style.background_gradient(subset=["score"])

Power: 95; largest change: 9.48e-13


,limit priority,score,ranking
Digital triage,0.221235,0.516214,1
Triage training,0.117682,0.274591,2
Extra shift,0.089655,0.209194,3


**Digital triage** has the highest network priority: its safety and capacity advantages outweigh its affordability disadvantage. Training is second and the extra shift is third. The comparison supports prioritizing one intervention under these assumptions; it does not estimate clinical outcomes or decide how several interventions should be combined.

## 4. References
---

- Saaty, T. L. (2004). [*Fundamentals of the analytic network process — Dependence and feedback in decision-making with a single network*](https://doi.org/10.1007/s11518-006-0158-y).
- Saaty, T. L., and Kułakowski, K. (2016). [*Axioms of the Analytic Hierarchy Process (AHP) and its Generalization to Dependence and Feedback: The Analytic Network Process (ANP)*](https://arxiv.org/abs/1605.05777).